In [1]:
import os
os.chdir(r"D:\Projects\Poverty Predictor Bd")

import pandas as pd
import geopandas as gpd
import json
import numpy as np
import shutil

LOVABLE_DIR = "lovable_app"
DATA_DIR    = f"{LOVABLE_DIR}/data"
BACKEND_DIR = f"{LOVABLE_DIR}/backend"

os.makedirs(DATA_DIR,    exist_ok=True)
os.makedirs(BACKEND_DIR, exist_ok=True)

print("Folders ready:")
print(f"  {LOVABLE_DIR}/")
print(f"  {DATA_DIR}/")
print(f"  {BACKEND_DIR}/")

Folders ready:
  lovable_app/
  lovable_app/data/
  lovable_app/backend/


In [2]:
df       = pd.read_csv("data/processed/master_features.csv")
shap_df  = pd.read_csv("data/processed/shap_values.csv")

# Merge SHAP values in
merged = df.merge(shap_df, on='district_name')

# Round all numbers
num_cols = merged.select_dtypes(include='number').columns
merged[num_cols] = merged[num_cols].round(4)

# Save
out_path = f"{DATA_DIR}/districts.json"
merged.to_json(out_path, orient='records', indent=2)

print(f"Saved: {out_path}")
print(f"Districts: {len(merged)}")
print(f"Columns: {len(merged.columns)}")
print(f"\nAll columns:")
for c in merged.columns:
    print(f"  {c}")

Saved: lovable_app/data/districts.json
Districts: 64
Columns: 50

All columns:
  district_name
  division_name
  ntl_mean_x
  ntl_std_x
  ntl_max_x
  ntl_min
  ntl_p25
  ntl_p75
  ntl_yoy_change_x
  ntl_trend_x
  pop_density_x
  pop_total
  ndvi_mean_x
  ndvi_std_x
  urban_fraction_x
  water_fraction_x
  elevation_mean_x
  elevation_std_x
  road_length_km
  poverty_hcr
  poverty_hcr_lower
  poverty_change
  ntl_per_capita_x
  area_sqkm
  road_density_x
  ntl_iqr_x
  ntl_mean_spatial_lag_x
  ntl_per_capita_spatial_lag_x
  ndvi_mean_spatial_lag_x
  pop_density_spatial_lag_x
  poverty_predicted
  ntl_mean_y
  ntl_std_y
  ntl_max_y
  ntl_per_capita_y
  ntl_trend_y
  ntl_yoy_change_y
  ntl_iqr_y
  pop_density_y
  ndvi_mean_y
  ndvi_std_y
  urban_fraction_y
  water_fraction_y
  elevation_mean_y
  elevation_std_y
  road_density_y
  ntl_mean_spatial_lag_y
  ntl_per_capita_spatial_lag_y
  ndvi_mean_spatial_lag_y
  pop_density_spatial_lag_y


In [4]:
# Cell 2 - Fixed
gdf = gpd.read_file("data/processed/master_features.gpkg")
df  = pd.read_csv("data/processed/master_features.csv")

# Add poverty_predicted from CSV into gdf
gdf['poverty_predicted'] = df['poverty_predicted'].values

# Keep only what frontend needs
gdf_simple = gdf[[
    'district_name', 'division_name',
    'poverty_hcr', 'ntl_mean',
    'poverty_predicted', 'geometry'
]].copy()

# Simplify geometry to reduce file size
gdf_simple['geometry'] = gdf_simple.geometry.simplify(
    tolerance=0.005, preserve_topology=True
)

# Save
out_path = f"{DATA_DIR}/bangladesh_districts.geojson"
gdf_simple.to_file(out_path, driver='GeoJSON')

size_kb = os.path.getsize(out_path) / 1024
print(f"Saved: {out_path}")
print(f"Size:  {size_kb:.1f} KB")
print(f"Districts: {len(gdf_simple)}")
print(f"Columns: {gdf_simple.columns.tolist()}")

Saved: lovable_app/data/bangladesh_districts.geojson
Size:  451.5 KB
Districts: 64
Columns: ['district_name', 'division_name', 'poverty_hcr', 'ntl_mean', 'poverty_predicted', 'geometry']


In [5]:
import joblib

model    = joblib.load("models/random_forest_final.pkl")
features = joblib.load("models/feature_cols.pkl")

# Feature importance
feat_imp = {
    feat: round(float(imp), 6)
    for feat, imp in zip(features, model.feature_importances_)
}
feat_imp_sorted = dict(
    sorted(feat_imp.items(), key=lambda x: -x[1])
)

model_info = {
    "model_name":    "Random Forest Regressor",
    "n_features":    len(features),
    "feature_names": features,
    "feature_importance": feat_imp_sorted,
    "performance": {
        "rmse": 3.626,
        "mae":  2.926,
        "r2":   0.241,
        "validation": "Leave-One-District-Out CV"
    },
    "models_comparison": [
        {"name": "Naive Baseline", "rmse": 4.163, 
         "mae": 3.596, "r2": 0.000},
        {"name": "CNN ResNet-18", "rmse": 4.354,  
         "mae": 3.188, "r2": -0.094},
        {"name": "Random Forest", "rmse": 3.626,  
         "mae": 2.926, "r2": 0.241},
    ],
    "division_results": [
        {"division": "Barishal",    "poverty_hcr": 26.9, 
         "n": 6,  "rf_rmse": 5.326, "cnn_rmse": 10.439},
        {"division": "Chattogram",  "poverty_hcr": 15.8, 
         "n": 11, "rf_rmse": 2.364, "cnn_rmse": 1.264},
        {"division": "Dhaka",       "poverty_hcr": 17.9, 
         "n": 13, "rf_rmse": 0.474, "cnn_rmse": 1.984},
        {"division": "Khulna",      "poverty_hcr": 14.8, 
         "n": 10, "rf_rmse": 4.180, "cnn_rmse": 1.020},
        {"division": "Mymensingh",  "poverty_hcr": 24.2, 
         "n": 4,  "rf_rmse": 5.622, "cnn_rmse": 3.636},
        {"division": "Rajshahi",    "poverty_hcr": 16.7, 
         "n": 8,  "rf_rmse": 2.039, "cnn_rmse": 2.630},
        {"division": "Rangpur",     "poverty_hcr": 24.8, 
         "n": 8,  "rf_rmse": 4.057, "cnn_rmse": 7.284},
        {"division": "Sylhet",      "poverty_hcr": 17.4, 
         "n": 4,  "rf_rmse": 2.514, "cnn_rmse": 1.685},
    ],
    "feature_ranges": {
        feat: {
            "min":    round(float(df[feat].min()), 4),
            "max":    round(float(df[feat].max()), 4),
            "median": round(float(df[feat].median()), 4),
            "mean":   round(float(df[feat].mean()), 4),
        }
        for feat in features
    },
    "slider_features": [
        {"key": "ntl_mean",       
         "label": "NTL Mean (Nighttime Light)", 
         "icon": "🌙", "step": 0.01},
        {"key": "ndvi_mean",      
         "label": "NDVI (Vegetation Index)",    
         "icon": "🌿", "step": 0.01},
        {"key": "elevation_mean", 
         "label": "Elevation Mean (m)",         
         "icon": "🏔️", "step": 0.5},
        {"key": "road_density",   
         "label": "Road Density (km/km²)",      
         "icon": "🛣️", "step": 0.1},
        {"key": "urban_fraction", 
         "label": "Urban Fraction",             
         "icon": "🏙️", "step": 0.001},
        {"key": "water_fraction", 
         "label": "Water Fraction",             
         "icon": "💧", "step": 0.001},
    ]
}

out_path = f"{DATA_DIR}/model_info.json"
with open(out_path, 'w') as f:
    json.dump(model_info, f, indent=2)

print(f"Saved: {out_path}")
print(f"Features: {len(features)}")
print(f"\nTop 5 features:")
for feat, imp in list(feat_imp_sorted.items())[:5]:
    print(f"  {feat}: {imp*100:.1f}%")

Saved: lovable_app/data/model_info.json
Features: 19

Top 5 features:
  elevation_mean: 19.2%
  ntl_mean_spatial_lag: 17.3%
  ntl_iqr: 9.3%
  road_density: 9.2%
  ntl_yoy_change: 6.5%


In [7]:
backend_code = '''from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import joblib
import numpy as np
import pandas as pd
import json
import os

app = FastAPI(
    title="Bangladesh Poverty Prediction API",
    description="Predicts district poverty using satellite features",
    version="1.0.0"
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

# ── Load model and data at startup ────────────────────────────
BASE = os.path.dirname(os.path.abspath(__file__))

model    = joblib.load(os.path.join(BASE, "models/random_forest_final.pkl"))
scaler   = joblib.load(os.path.join(BASE, "models/scaler_final.pkl"))
features = joblib.load(os.path.join(BASE, "models/feature_cols.pkl"))
df       = pd.read_csv(os.path.join(BASE, "data/master_features.csv"))

with open(os.path.join(BASE, "data/model_info.json")) as f:
    model_info = json.load(f)

# ── Input schema ──────────────────────────────────────────────
class PredictionInput(BaseModel):
    ntl_mean:       float
    ndvi_mean:      float
    elevation_mean: float
    road_density:   float
    urban_fraction: float
    water_fraction: float

# ── Routes ────────────────────────────────────────────────────
@app.get("/")
def root():
    return {
        "status": "ok",
        "model":  "Random Forest Poverty Predictor",
        "docs":   "/docs"
    }

@app.get("/health")
def health():
    return {"status": "healthy", "districts": len(df)}

@app.post("/predict")
def predict(inp: PredictionInput):
    # Build full feature vector using medians for non-slider features
    input_dict = {f: float(df[f].median()) for f in features}
    input_dict.update(inp.dict())

    X        = pd.DataFrame([input_dict])[features]
    X_scaled = scaler.transform(X)
    pred     = float(np.clip(model.predict(X_scaled)[0], 14.8, 26.9))

    # Label
    if pred > 23:
        label, color = "High Poverty",     "#ef4444"
    elif pred > 19:
        label, color = "Moderate Poverty", "#f59e0b"
    else:
        label, color = "Lower Poverty",    "#22c55e"

    # Similar districts
    df_copy = df.copy()
    df_copy["_dist"] = (df_copy["poverty_predicted"] - pred).abs()
    similar = df_copy.nsmallest(3, "_dist")[[
        "district_name", "division_name", "poverty_hcr"
    ]].to_dict("records")

    return {
        "prediction":        round(pred, 2),
        "label":             label,
        "color":             color,
        "similar_districts": similar
    }

@app.get("/districts")
def get_districts():
    cols = [
        "district_name", "division_name",
        "poverty_hcr", "poverty_predicted",
        "ntl_mean", "ndvi_mean", "elevation_mean",
        "road_density", "urban_fraction", "water_fraction",
        "ntl_per_capita", "ntl_trend", "ntl_iqr",
        "water_fraction", "elevation_std"
    ]
    return df[cols].round(3).to_dict("records")

@app.get("/model-info")
def get_model_info():
    return model_info

@app.get("/district/{district_name}")
def get_district(district_name: str):
    row = df[df["district_name"] == district_name]
    if len(row) == 0:
        return {"error": f"District {district_name} not found"}
    return row.iloc[0].to_dict()
'''

with open(f"{BACKEND_DIR}/main.py", 'w',encoding='utf-8') as f:
    f.write(backend_code)

print(f"Saved: {BACKEND_DIR}/main.py")

Saved: lovable_app/backend/main.py


In [8]:
# requirements.txt for backend
requirements = """fastapi==0.109.0
uvicorn==0.27.0
scikit-learn==1.8.0
joblib==1.5.3
numpy==1.26.4
pandas==2.2.0
python-multipart==0.0.9
"""

with open(f"{BACKEND_DIR}/requirements.txt", 'w') as f:
    f.write(requirements)

# Render deployment config
render_config = """services:
  - type: web
    name: bangladesh-poverty-api
    runtime: python
    rootDir: lovable_app/backend
    buildCommand: pip install -r requirements.txt
    startCommand: uvicorn main:app --host 0.0.0.0 --port $PORT
    envVars:
      - key: PYTHON_VERSION
        value: 3.10.0
"""

with open("render.yaml", 'w') as f:
    f.write(render_config)

# .gitignore addition for lovable
print(f"Saved: {BACKEND_DIR}/requirements.txt")
print(f"Saved: render.yaml")

Saved: lovable_app/backend/requirements.txt
Saved: render.yaml


In [9]:
# Models
os.makedirs(f"{BACKEND_DIR}/models", exist_ok=True)
for fname in ["random_forest_final.pkl", 
              "scaler_final.pkl", 
              "feature_cols.pkl"]:
    src = f"models/{fname}"
    dst = f"{BACKEND_DIR}/models/{fname}"
    shutil.copy(src, dst)
    size = os.path.getsize(dst)/1024
    print(f"Copied: {fname} ({size:.0f} KB)")

# Data
os.makedirs(f"{BACKEND_DIR}/data", exist_ok=True)
for fname in ["master_features.csv", "shap_values.csv"]:
    src = f"data/processed/{fname}"
    dst = f"{BACKEND_DIR}/data/{fname}"
    shutil.copy(src, dst)
    size = os.path.getsize(dst)/1024
    print(f"Copied: {fname} ({size:.0f} KB)")

# Copy model_info.json
shutil.copy(
    f"{DATA_DIR}/model_info.json",
    f"{BACKEND_DIR}/data/model_info.json"
)
print(f"Copied: model_info.json")

Copied: random_forest_final.pkl (335 KB)
Copied: scaler_final.pkl (1 KB)
Copied: feature_cols.pkl (0 KB)
Copied: master_features.csv (19 KB)
Copied: shap_values.csv (25 KB)
Copied: model_info.json


In [10]:
import subprocess
import time
import requests

print("Starting FastAPI server...")
print("If this hangs, open a NEW terminal and run:")
print(f"  cd 'D:\\Projects\\Poverty Predictor Bd\\{BACKEND_DIR}'")
print(f"  uvicorn main:app --reload --port 8000")
print(f"\nThen visit: http://localhost:8000/docs")
print(f"\nTest the prediction endpoint:")
print("""
import requests
resp = requests.post("http://localhost:8000/predict", json={
    "ntl_mean": 0.66,
    "ndvi_mean": 0.59,
    "elevation_mean": 11.88,
    "road_density": 2.13,
    "urban_fraction": 0.01,
    "water_fraction": 0.05
})
print(resp.json())
""")

Starting FastAPI server...
If this hangs, open a NEW terminal and run:
  cd 'D:\Projects\Poverty Predictor Bd\lovable_app/backend'
  uvicorn main:app --reload --port 8000

Then visit: http://localhost:8000/docs

Test the prediction endpoint:

import requests
resp = requests.post("http://localhost:8000/predict", json={
    "ntl_mean": 0.66,
    "ndvi_mean": 0.59,
    "elevation_mean": 11.88,
    "road_density": 2.13,
    "urban_fraction": 0.01,
    "water_fraction": 0.05
})
print(resp.json())



In [11]:
print("="*55)
print("LOVABLE APP PREPARATION COMPLETE")
print("="*55)

all_files = []
for root, dirs, files in os.walk(LOVABLE_DIR):
    for file in files:
        path = os.path.join(root, file)
        size = os.path.getsize(path)/1024
        all_files.append((path, size))
        print(f"  {path:55} {size:8.1f} KB")

total = sum(s for _, s in all_files)
print(f"\nTotal: {len(all_files)} files, {total/1024:.1f} MB")
print(f"\nNext steps:")
print(f"  1. Test backend locally (see Cell 7)")
print(f"  2. Push lovable_app/ to GitHub")
print(f"  3. Deploy backend on render.com")  
print(f"  4. Go to lovable.dev with the prompt")

LOVABLE APP PREPARATION COMPLETE
  lovable_app\backend\main.py                                  3.5 KB
  lovable_app\backend\requirements.txt                         0.1 KB
  lovable_app\backend\data\master_features.csv                19.1 KB
  lovable_app\backend\data\model_info.json                     5.9 KB
  lovable_app\backend\data\shap_values.csv                    25.2 KB
  lovable_app\backend\models\feature_cols.pkl                  0.3 KB
  lovable_app\backend\models\random_forest_final.pkl         334.6 KB
  lovable_app\backend\models\scaler_final.pkl                  1.3 KB
  lovable_app\data\bangladesh_districts.geojson              451.5 KB
  lovable_app\data\districts.json                             93.3 KB
  lovable_app\data\model_info.json                             5.9 KB

Total: 11 files, 0.9 MB

Next steps:
  1. Test backend locally (see Cell 7)
  2. Push lovable_app/ to GitHub
  3. Deploy backend on render.com
  4. Go to lovable.dev with the prompt


In [12]:
import requests

# Test health
r = requests.get("http://localhost:8000/health")
print("Health:", r.json())

# Test prediction
r = requests.post("http://localhost:8000/predict", json={
    "ntl_mean":       0.66,
    "ndvi_mean":      0.59,
    "elevation_mean": 11.88,
    "road_density":   2.13,
    "urban_fraction": 0.01,
    "water_fraction": 0.05
})
print("\nPrediction:", r.json())

# Test districts
r = requests.get("http://localhost:8000/districts")
districts = r.json()
print(f"\nDistricts loaded: {len(districts)}")
print(f"First district:   {districts[0]['district_name']}")

# Test model info
r = requests.get("http://localhost:8000/model-info")
info = r.json()
print(f"\nModel info loaded: {info['model_name']}")
print(f"Top feature: {list(info['feature_importance'].keys())[0]}")

Health: {'status': 'healthy', 'districts': 64}

Prediction: {'prediction': 16.71, 'label': 'Lower Poverty', 'color': '#22c55e', 'similar_districts': [{'district_name': 'Jashore', 'division_name': 'Khulna', 'poverty_hcr': 14.8}, {'district_name': 'Cumilla', 'division_name': 'Chattogram', 'poverty_hcr': 15.8}, {'district_name': 'Bogura', 'division_name': 'Rajshahi', 'poverty_hcr': 16.7}]}

Districts loaded: 64
First district:   Barguna

Model info loaded: Random Forest Regressor
Top feature: elevation_mean


In [13]:
import requests

BASE = "https://bangladesh-poverty-api.onrender.com"

# Test health
r = requests.get(f"{BASE}/health")
print("Health:", r.json())

# Test prediction
r = requests.post(f"{BASE}/predict", json={
    "ntl_mean": 0.66,
    "ndvi_mean": 0.59,
    "elevation_mean": 11.88,
    "road_density": 2.13,
    "urban_fraction": 0.01,
    "water_fraction": 0.05
})
print("Prediction:", r.json())

Health: {'status': 'healthy', 'districts': 64}
Prediction: {'prediction': 16.71, 'label': 'Lower Poverty', 'color': '#22c55e', 'similar_districts': [{'district_name': 'Jashore', 'division_name': 'Khulna', 'poverty_hcr': 14.8}, {'district_name': 'Cumilla', 'division_name': 'Chattogram', 'poverty_hcr': 15.8}, {'district_name': 'Bogura', 'division_name': 'Rajshahi', 'poverty_hcr': 16.7}]}
